In [1]:
import bw2io as bi
import bw2data as bd
import bw2calc as bc
import pandas as pd
from bw2io import SimaProCSVImporter

In [2]:
bd.projects.set_current('Practicas')

In [3]:
list(bd.projects) #the prefix "bw" indicates that "projects" is a method of the bw2data package

[Project: default, Project: Practicas, Project: ECO]

In [4]:
bi.bw2setup()

Biosphere database already present!!! No setup is needed


In [5]:
list(bd.databases) # check if there are databases in the project, and how they are named. 

['biosphere3', 'BAFU-2025 Version 2 - openLCA 2026-03-09']

In [6]:
imp = SimaProCSVImporter(r'C:\Users\WorkGroup\Desktop\bd\ecoinvent 3.12 Cutoff Unit 2025-12-19.csv')
imp.match_database(fields=("name","unit", "location"))
imp.apply_strategies()
imp.statistics()

Extracted 9842 unallocated datasets in 13.14 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: assign_only_product_as_production
Applying strategy: drop_unspecified_subcategories
Applying strategy: sp_allocate_products
Applying strategy: fix_zero_allocation_products
Applying strategy: split_simapro_name_geo
Applying strategy: strip_biosphere_exc_locations
Applying strategy: migrate_datasets
Applying strategy: migrate_exchanges
Applying strategy: set_code_by_activity_hash
Applying strategy: link_technosphere_based_on_name_unit_location
Applying strategy: change_electricity_unit_mj_to_kwh
Applying strategy: set_lognormal_loc_value_uncertainty_safe
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_simapro_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: normalize_simapro_biosphere_names
Applying strategy: migrate_exchanges
Ap

(9842, 428665, 108412)

In [7]:
index = {}

for ds in imp.data:
    key = (
        ds.get("name", "").lower().strip(),
        ds.get("location"),
        ds.get("unit")
    )
    index.setdefault(key, []).append(ds)

In [8]:
for ds in imp.data:
    for exc in ds.get("exchanges", []):
        
        if exc.get("type") == "technosphere":
            
            key = (
                exc.get("name", "").lower().strip(),
                exc.get("location"),
                exc.get("unit")
            )
            
            candidates = index.get(key, [])
            
            if len(candidates) == 1:
                target = candidates[0]
                exc["input"] = (target["database"], target["code"])

In [9]:
def fuzzy_key(name):
    return name.lower().split(",")[0].strip()

index_simple = {}

for ds in imp.data:
    key = fuzzy_key(ds.get("name", ""))
    index_simple.setdefault(key, []).append(ds)

In [10]:
for ds in imp.data:
    for exc in ds.get("exchanges", []):
        
        if exc.get("type") == "technosphere" and "input" not in exc:
            
            key = fuzzy_key(exc.get("name", ""))
            candidates = index_simple.get(key, [])
            
            if len(candidates) == 1:
                target = candidates[0]
                exc["input"] = (target["database"], target["code"])

In [11]:
imp.match_database(fields=("name", "unit", "location"))
imp.statistics()

Applying strategy: link_iterable_by_fields
Couldn't apply strategy link_iterable_by_fields:
	Object in source database can't be uniquely linked to target database.
Problematic dataset         is:
{'filename': '(missing)',
 'location': '(missing)',
 'name': 'Polyacrylonitrile production (PAN) by polymerisation {RER} {RER} | '
         'Polyacrylonitrile production (PAN) by polymerisation , U',
 'unit': 'kilogram'}
Possible targets include (at least one not shown):
[{'filename': 'C:\\Users\\WorkGroup\\Desktop\\bd\\BAFU-2025 Version 2 - '
              'openLCA 2026-03-09.csv',
  'location': '(missing)',
  'name': 'Polyacrylonitrile production (PAN), by polymerisation {RER} {RER} | '
          'Polyacrylonitrile production (PAN), by polymerisation , U',
  'unit': 'kilogram'}]
11747 datasets
416293 exchanges
71606 unlinked exchanges
  Type biosphere: 578 unique unlinked exchanges
  Type production: 9356 unique unlinked exchanges
  Type technosphere: 918 unique unlinked exchanges


(11747, 416293, 71606)

In [12]:
import re

def normalize(name):
    name = name.lower()
    name = re.sub(r"\{.*?\}", "", name)   # quitar {RER}
    name = re.sub(r"\|.*", "", name)      # quitar después de |
    name = name.replace(",", "")          # quitar comas
    name = name.replace("  ", " ")
    name = name.strip()
    return name

# aplicar a datasets
for ds in imp.data:
    ds["name"] = normalize(ds.get("name", ""))

for ds in imp.data:
    for exc in ds.get("exchanges", []):
        exc["name"] = normalize(exc.get("name", ""))

In [13]:
imp.match_database(fields=("name", "unit", "location"))
imp.statistics()

Applying strategy: link_iterable_by_fields
Couldn't apply strategy link_iterable_by_fields:
	Object in source database can't be uniquely linked to target database.
Problematic dataset         is:
{'filename': '(missing)',
 'location': '(missing)',
 'name': 'polyacrylonitrile production (pan) by polymerisation',
 'unit': 'kilogram'}
Possible targets include (at least one not shown):
[{'filename': 'C:\\Users\\WorkGroup\\Desktop\\bd\\BAFU-2025 Version 2 - '
              'openLCA 2026-03-09.csv',
  'location': '(missing)',
  'name': 'polyacrylonitrile production (pan) by polymerisation',
  'unit': 'kilogram'}]
11747 datasets
416293 exchanges
71606 unlinked exchanges
  Type biosphere: 578 unique unlinked exchanges
  Type production: 6528 unique unlinked exchanges
  Type technosphere: 401 unique unlinked exchanges


(11747, 416293, 71606)

In [14]:
for i, ds in enumerate(imp.data):
    ds["code"] = f"{ds.get('name', 'process')}_{i}"

In [15]:
imp.match_database(fields=("name", "unit", "location"))
imp.statistics()

Applying strategy: link_iterable_by_fields
Couldn't apply strategy link_iterable_by_fields:
	Object in source database can't be uniquely linked to target database.
Problematic dataset         is:
{'filename': '(missing)',
 'location': '(missing)',
 'name': 'polyacrylonitrile production (pan) by polymerisation',
 'unit': 'kilogram'}
Possible targets include (at least one not shown):
[{'filename': 'C:\\Users\\WorkGroup\\Desktop\\bd\\BAFU-2025 Version 2 - '
              'openLCA 2026-03-09.csv',
  'location': '(missing)',
  'name': 'polyacrylonitrile production (pan) by polymerisation',
  'unit': 'kilogram'}]
11747 datasets
416293 exchanges
71606 unlinked exchanges
  Type biosphere: 578 unique unlinked exchanges
  Type production: 6528 unique unlinked exchanges
  Type technosphere: 401 unique unlinked exchanges


(11747, 416293, 71606)

In [16]:
bad = []

for ds in imp.data:
    for exc in ds.get("exchanges", []):
        if "input" not in exc or "amount" not in exc:
            bad.append(exc)

len(bad)

71606

In [17]:
# limpiar
for ds in imp.data:
    ds["exchanges"] = [
        exc for exc in ds.get("exchanges", [])
        if "input" in exc and "amount" in exc
    ]

# guardar


In [18]:
valid_codes = {ds["code"] for ds in imp.data}
db_name = imp.db_name  # normalmente ya existe

In [19]:
valid_codes = {ds["code"] for ds in imp.data}
db_name = imp.db_name

for ds in imp.data:
    clean_excs = []
    
    for exc in ds.get("exchanges", []):
        
        if "amount" not in exc or "input" not in exc:
            continue
        
        input_db, input_code = exc["input"]
        
        # 🔥 conservar biosphere SIEMPRE
        if input_db == "biosphere3":
            clean_excs.append(exc)
            continue
        
        # validar solo technosphere
        if input_code in valid_codes:
            clean_excs.append(exc)
    
    ds["exchanges"] = clean_excs

In [20]:
imp.write_database()

Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:12


Title: Writing activities to SQLite3 database:
  Started: 03/24/2026 13:53:10
  Finished: 03/24/2026 13:53:22
  Total time elapsed: 00:00:12
  CPU %: 99.50
  Memory %: 5.27
Created database: BAFU-2025 Version 2 - openLCA 2026-03-09


Brightway2 SQLiteBackend: BAFU-2025 Version 2 - openLCA 2026-03-09

In [21]:
total_excs = 0

for ds in imp.data:
    total_excs += len(ds.get("exchanges", []))

total_excs

249614